# MMF GPU Model Runner
Runs a single GPU model (global or foundation). Called by orchestrator notebooks via `dbutils.notebook.run()`.

In [ ]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("model", "")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("use_case", "")
dbutils.widgets.text("train_table", "")
dbutils.widgets.text("freq", "")
dbutils.widgets.text("prediction_length", "")
dbutils.widgets.text("backtest_length", "")
dbutils.widgets.text("stride", "")
dbutils.widgets.text("metric", "smape")
dbutils.widgets.text("group_id", "unique_id")
dbutils.widgets.text("date_col", "ds")
dbutils.widgets.text("target", "y")
dbutils.widgets.text("num_nodes", "1")
# Covariates — see Skill 4 Feature Type Decision Guide.
# SAFE:  static_features, dynamic_historical_*. AVOID: dynamic_future_* (requires scoring table
# with known future values for every future ds; if missing, pipeline errors or drops series).
dbutils.widgets.text("static_features", "")
dbutils.widgets.text("dynamic_future_numerical", "")
dbutils.widgets.text("dynamic_future_categorical", "")
dbutils.widgets.text("dynamic_historical_numerical", "")
dbutils.widgets.text("dynamic_historical_categorical", "")
# Scoring table: always blank unless dynamic_future_* is in use (not recommended).
dbutils.widgets.text("scoring_table", "")

In [ ]:
import subprocess, sys

model_name = dbutils.widgets.get("model")
is_global = "NeuralForecast" in model_name
is_timesfm = "TimesFM" in model_name

MMF_GIT = "mmf_sa @ git+https://github.com/databricks-industry-solutions/many-model-forecasting.git@v0.1.2"
TIMESFM_TARBALL = "https://github.com/google-research/timesfm/archive/2dcc66fbfe2155adba1af66aa4d564a0ee52f61e.tar.gz"

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", *args, "--quiet"])

if is_global:
    pip(f"mmf_sa[global] @ git+https://github.com/databricks-industry-solutions/many-model-forecasting.git@v0.1.2")
elif is_timesfm:
    pip(MMF_GIT)
    pip(f"timesfm[torch] @ {TIMESFM_TARBALL}")
    pip("utilsforecast==0.2.15")
else:
    pip(MMF_GIT)
    pip("chronos-forecasting==2.2.2", "utilsforecast==0.2.15")

dbutils.library.restartPython()

In [ ]:
def _parse_col_list(s):
    if not s or not str(s).strip():
        return []
    return [x.strip() for x in str(s).split(",") if x.strip()]


catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
model = dbutils.widgets.get("model")
run_id = dbutils.widgets.get("run_id")
use_case = dbutils.widgets.get("use_case")
train_table = dbutils.widgets.get("train_table")
freq = dbutils.widgets.get("freq")
prediction_length = int(dbutils.widgets.get("prediction_length"))
backtest_length = int(dbutils.widgets.get("backtest_length"))
stride = int(dbutils.widgets.get("stride"))
metric = dbutils.widgets.get("metric")
group_id = dbutils.widgets.get("group_id")
date_col = dbutils.widgets.get("date_col")
target = dbutils.widgets.get("target")
num_nodes = int(dbutils.widgets.get("num_nodes"))
scoring_table = (dbutils.widgets.get("scoring_table") or "").strip()
static_features = _parse_col_list(dbutils.widgets.get("static_features"))
dynamic_future_numerical = _parse_col_list(dbutils.widgets.get("dynamic_future_numerical"))
dynamic_future_categorical = _parse_col_list(dbutils.widgets.get("dynamic_future_categorical"))
dynamic_historical_numerical = _parse_col_list(dbutils.widgets.get("dynamic_historical_numerical"))
dynamic_historical_categorical = _parse_col_list(dbutils.widgets.get("dynamic_historical_categorical"))

In [ ]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("py4j.clientserver").setLevel(logging.ERROR)
logging.getLogger("py4j.java_gateway").setLevel(logging.ERROR)

from mmf_sa import run_forecast

user = spark.sql("SELECT current_user() AS user").collect()[0]["user"]


def _exog_kwargs():
    kw = {}
    if static_features:
        kw["static_features"] = static_features
    if dynamic_future_numerical:
        kw["dynamic_future_numerical"] = dynamic_future_numerical
    if dynamic_future_categorical:
        kw["dynamic_future_categorical"] = dynamic_future_categorical
    if dynamic_historical_numerical:
        kw["dynamic_historical_numerical"] = dynamic_historical_numerical
    if dynamic_historical_categorical:
        kw["dynamic_historical_categorical"] = dynamic_historical_categorical
    return kw


train_fqn = f"{catalog}.{schema}.{train_table}"
scoring_fqn = train_fqn if not scoring_table else f"{catalog}.{schema}.{scoring_table}"

run_forecast(
    spark=spark,
    train_data=train_fqn,
    scoring_data=scoring_fqn,
    scoring_output=f"{catalog}.{schema}.{use_case}_scoring_output",
    evaluation_output=f"{catalog}.{schema}.{use_case}_evaluation_output",
    model_output=f"{catalog}.{schema}",
    group_id=group_id,
    date_col=date_col,
    target=target,
    freq=freq,
    prediction_length=prediction_length,
    backtest_length=backtest_length,
    stride=stride,
    metric=metric,
    train_predict_ratio=1,
    data_quality_check=True,
    resample=False,
    active_models=[model],
    experiment_path=f"/Users/{user}/mmf/{use_case}",
    use_case_name=use_case,
    run_id=run_id,
    accelerator="gpu",
    num_nodes=num_nodes,
    **_exog_kwargs(),
)